In [ ]:
import logging
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger("sae")
logger.setLevel(logging.INFO)


def repo_root() -> Path:
    """Return the repository root containing downloaded artifact directories."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts" / "download_artifact.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root. Run this notebook from inside the repository.")


REPO_ROOT = repo_root()
DATA_MODEL_WEIGHTS_DIR = REPO_ROOT / "data_model_weights"
MNIST_DATA_DIR = REPO_ROOT / "external" / "mnist"


class SAE_l2w_encoder_decoder(nn.Module):
    """
    Sparse Autoencoder with untied weights (decoder is a separate nn.Linear) and L2 regularization on weights.

    - Input is flattened to (N, D).
    - ReLU enforces non-negativity in the code (common for sparse coding).
    - Loss = MSE(recon, x) + l1 * mean(|z|).
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        l1: float = 1e-3,
        l2_w: float = 1e-3,
        bias: bool = False,
        seed: int =0,
    ) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l2_w = float(l2_w)
        self.seed = int(seed)

        # Encoder parameters
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=bias)
        # Decoder: untied weights (new layer)
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=bias)

        # Initialize with reproducible seed
        gen = torch.Generator()
        gen.manual_seed(self.seed)

        # Kaiming init for ReLU
        nn.init.kaiming_uniform_(self.encoder.weight, a=0.0, generator=gen)
        if self.encoder.bias is not None:
            nn.init.zeros_(self.encoder.bias)
        nn.init.kaiming_uniform_(self.decoder.weight, a=0.0, generator=gen)
        if self.decoder.bias is not None:
            nn.init.zeros_(self.decoder.bias)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        z = F.relu(self.encoder(x))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, x_hat: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        recon = F.mse_loss(x_hat, x)
        sparsity = z.abs().mean()
        return recon + self.l1 * sparsity + self.l2_w * (self.decoder.weight.pow(2).sum()  + self.encoder.weight.pow(2).sum())
    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

    def fit(
        self,
        loader: DataLoader,
        epochs: int = 50,
        lr: float = 1e-3,
        weight_decay: float = 0.0,
        device: Optional[torch.device] = None,
        log_interval: int = 100,
    ) -> None:
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)

        step = 0
        self.train()  # use the standard nn.Module.train mode
        for epoch in range(1, epochs + 1):
            for xb, _ in loader:  # targets are xb itself (autoencoder)
                xb = xb.to(device, non_blocking=True)
                x_hat, z = self(xb)
                sparsity = (z != 0).sum().item()
                loss = self.loss(xb, x_hat, z)

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

                if step % log_interval == 0:
                    print(f"epoch={epoch} step={step} loss={loss.item():.6f} sparsity={sparsity}")
                step += 1

In [ ]:
import logging
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

logger = logging.getLogger("sae")
logger.setLevel(logging.INFO)


class SAE_l1w_encoder_decoder(nn.Module):
    """
    Sparse Autoencoder with untied weights (decoder is a separate nn.Linear) and L2 regularization on weights.

    - Input is flattened to (N, D).
    - ReLU enforces non-negativity in the code (common for sparse coding).
    - Loss = MSE(recon, x) + l1 * mean(|z|).
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        l1: float = 1e-3,
        l1_w: float = 1e-3,
        bias: bool = False,
        seed: int = 0,
    ) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.l1 = float(l1)
        self.l1_w = float(l1_w)
        self.seed = int(seed)

        # Encoder parameters
        self.encoder = nn.Linear(self.input_dim, self.hidden_dim, bias=bias)
        # Decoder: untied weights (new layer)
        self.decoder = nn.Linear(self.hidden_dim, self.input_dim, bias=bias)

        # Initialize with reproducible seed
        gen = torch.Generator()
        gen.manual_seed(self.seed)
        # Kaiming init for ReLU
        nn.init.kaiming_uniform_(self.encoder.weight, a=0.0, generator=gen)
        if self.encoder.bias is not None:
            nn.init.zeros_(self.encoder.bias)
        nn.init.kaiming_uniform_(self.decoder.weight, a=0.0, generator=gen)
        if self.decoder.bias is not None:
            nn.init.zeros_(self.decoder.bias)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        z = F.relu(self.encoder(x))
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, x_hat: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
        recon = F.mse_loss(x_hat, x)
        sparsity = z.abs().mean()
        return recon + self.l1 * sparsity + self.l1_w * (self.decoder.weight.abs().mean() + self.encoder.weight.abs().mean())

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

    def fit(
        self,
        loader: DataLoader,
        epochs: int = 50,
        lr: float = 1e-3,
        weight_decay: float = 0.0,
        device: Optional[torch.device] = None,
        log_interval: int = 100,
    ) -> None:
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(device)
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)

        step = 0
        self.train()  # use the standard nn.Module.train mode
        for epoch in range(1, epochs + 1):
            for xb, _ in loader:  # targets are xb itself (autoencoder)
                xb = xb.to(device, non_blocking=True)
                x_hat, z = self(xb)
                sparsity = (z != 0).sum().item()
                loss = self.loss(xb, x_hat, z)

                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

                if step % log_interval == 0:
                    print(f"epoch={epoch} step={step} loss={loss.item():.6f} sparsity={sparsity}")
                step += 1

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# Load MNIST data as input and target (unsupervised)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),  # gives [0,1] float32, shape=[1,28,28]
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to [784]
])

mnist_train = datasets.MNIST(root=str(MNIST_DATA_DIR), train=True, download=True, transform=mnist_transform)

def ae_collate(batch):
    xs = torch.stack([x for x, _ in batch])
    return xs, xs

loader = DataLoader(
    mnist_train,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=ae_collate,
)

input_dim = 28 * 28

# Save models where the artifact downloader restores MNIST model weights.
output_dir = DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs_not_constrain" / "trained_autoencoder_MNISTs_not_constrain_squared" / "trained_autoencoder_MNISTs_not_constrain_squared"
output_dir.mkdir(parents=True, exist_ok=True)

# Hyperparameter sweep for l1_w and l1
l1_values = [1e-1]
l2_w_values = [1e-5]
seeds = [0,1,2]
autoencoders = {}


for seed in seeds:
    for l1 in l1_values:
        for l2_w in l2_w_values:
            # unique model name for each sweep setting
            model_name = f"sae_l1w_l1_{l1}_l2w_{l2_w}_100ep_seed_{seed}"
            ae = SAE_l2w_encoder_decoder(
                seed=seed,
                input_dim=input_dim, 
                hidden_dim=2 * input_dim, 
                l1=l1, 
                l2_w=l2_w,
            )
            print(f"Training {model_name}")
            ae.fit(
                loader=loader,
                epochs=100,
                lr=1e-3,
                weight_decay=0.0,
                log_interval=100,
            )
            autoencoders[model_name] = ae

            # Save the trained model
            save_path = output_dir / f"{model_name}.pt"
            torch.save(ae, save_path)
            print(f"Saved model to {save_path}")



In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# Load MNIST data as input and target (unsupervised)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),  # gives [0,1] float32, shape=[1,28,28]
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to [784]
])

mnist_train = datasets.MNIST(root=str(MNIST_DATA_DIR), train=True, download=True, transform=mnist_transform)

def ae_collate(batch):
    xs = torch.stack([x for x, _ in batch])
    return xs, xs

loader = DataLoader(
    mnist_train,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=ae_collate,
)

input_dim = 28 * 28

# Save models where the artifact downloader restores MNIST model weights.
output_dir = DATA_MODEL_WEIGHTS_DIR / "trained_autoencoder_MNISTs_not_constrain" / "trained_autoencoder_MNISTs_not_constrain_squared" / "trained_autoencoder_MNISTs_not_constrain"
output_dir.mkdir(parents=True, exist_ok=True)

# Hyperparameter sweep for l1_w and l1
l1_values = [1e-1]
l1_w_values = [0, 1e-3]
seeds = [0, 1, 2]
autoencoders = {}


for seed in seeds:
    for l1 in l1_values:
        for l1_w in l1_w_values:
            # unique model name for each sweep setting
            model_name = f"sae_l1w_l1_{l1}_l1w_{l1_w}_100ep_seed_{seed}"
            ae = SAE_l1w_encoder_decoder(
                seed=seed,
                input_dim=input_dim, 
                hidden_dim=2 * input_dim, 
                l1=l1, 
                l1_w=l1_w,
            )
            print(f"Training {model_name}")
            ae.fit(
                loader=loader,
                epochs=100,
                lr=1e-3,
                weight_decay=0.0,
                log_interval=100,
            )
            autoencoders[model_name] = ae

            # Save the trained model
            save_path = output_dir / f"{model_name}.pt"
            torch.save(ae, save_path)
            print(f"Saved model to {save_path}")

